In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
merged_df = pd.read_parquet("../Model_Data/merged_df_50Hz.parquet")

In [3]:

# Deine IDs pro Klasse
ids_per_class = merged_df.groupby("Tag")["ID"].unique().to_dict()

train_ids = []
test_ids = []

for tag, ids in ids_per_class.items():
    train, test = train_test_split(ids, test_size=0.2, random_state=42)
    train_ids.extend(train)
    test_ids.extend(test)

# Train / Test DataFrames
df_train = merged_df[merged_df["ID"].isin(train_ids)].copy()
df_test  = merged_df[merged_df["ID"].isin(test_ids)].copy()

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (883164, 16)
Test shape: (216729, 16)


In [4]:
def create_sliding_windows_full(df, feature_cols, label_col="Tag",
                                window_size=100, step_size=50):

    
    X = []
    y = []

    for id_, group in df.groupby("ID"):
        group = group.sort_values("time_elapsed")
        data = group[feature_cols].values
        label = group[label_col].iloc[0]  # eine Bewegung pro Aufnahme
        
        for start in range(0, len(group) - window_size + 1, step_size):
            end = start + window_size
            window = data[start:end]
            
            X.append(window)
            y.append(label)
    
    X = np.array(X)
    y = np.array(y)
    return X, y

In [5]:
feature_cols = [
    "x_acc", "y_acc", "z_acc",
    "x_gyr", "y_gyr", "z_gyr",
    "qx", "qy", "qz", "qw",
    "roll", "pitch", "yaw"
]

WINDOW_SIZE = 100
STEP_SIZE = 50

X_train, y_train = create_sliding_windows_full(df_train, feature_cols,
                                               window_size=WINDOW_SIZE,
                                               step_size=STEP_SIZE)

X_test, y_test = create_sliding_windows_full(df_test, feature_cols,
                                             window_size=WINDOW_SIZE,
                                             step_size=STEP_SIZE)

In [18]:
np.savez_compressed("../Model_data/train.npz", X=X_train, y=y_train)
np.savez_compressed("../Model_data/test.npz", X=X_test, y=y_test)